In [1]:
import numpy as np
import os
import joblib
from sklearn.ensemble import RandomForestClassifier

class XRayFusionEngine:
    def __init__(self):
        self.save_dir = "deployment/fusion_assets"
        os.makedirs(self.save_dir, exist_ok=True)
        self.classes = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia',
                        'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening',
                        'Hernia', 'No Finding']
        # Light max_depth ensures fast inference and prevents overfitting synthetic data
        self.model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)

    def train_model(self):
        print("⚙️ Training X-Ray Clinical Reasoner...")
        samples = 2000
        # Inputs: 14 Vision Logits + Age + Gender + Prev_Diag
        X_vision = np.random.rand(samples, len(self.classes))
        X_clinical = np.column_stack((
            np.random.randint(18, 90, samples),
            np.random.randint(0, 2, samples),
            np.random.randint(0, 2, samples)
        ))
        X = np.hstack((X_vision, X_clinical))
        y = np.argmax(X_vision, axis=1)

        # Clinical Rule: Older patients with history are biased towards pulmonary conditions
        for i in range(samples):
            if X_clinical[i][0] > 65 and X_clinical[i][2] == 1:
                if y[i] == 14: y[i] = 2 # Change 'No Finding' to 'Effusion' bias

        self.model.fit(X, y)
        joblib.dump(self.model, os.path.join(self.save_dir, "xray_fusion.joblib"))
        print("✅ X-Ray Fusion Model Saved.")

    def generate_report(self, vision_probs, age, gender, prev_diag):
        top_idx = np.argmax(vision_probs)
        condition = self.classes[top_idx]
        base_conf = vision_probs[top_idx]

        multiplier = 1.0
        reasoning = []
        reasoning.append(f"1. Radiographic CNN Analysis: The primary vision engine detected interstitial/alveolar opacities consistent with {condition} (Baseline AI Confidence: {base_conf*100:.1f}%).")

        if age > 60 and condition in ['Effusion', 'Edema', 'Pneumonia', 'Atelectasis']:
            multiplier += 0.15
            reasoning.append(f"2. Demographic Correlation: Patient age ({age}) indicates reduced baseline pulmonary compliance and higher susceptibility to fluid retention or lower respiratory tract infections.")

        if prev_diag:
            multiplier += 0.20
            reasoning.append(f"3. Longitudinal Risk Factor: Positive history of cardiopulmonary issues significantly elevates the pre-test probability of the current radiographic findings being acute.")

        final_conf = min(base_conf * multiplier, 0.99)
        reasoning.append(f"► Final Assessment (Weighted Conf: {final_conf*100:.1f}%): High suspicion for {condition}. As an assistive diagnostic tool, it is highly recommended the attending radiologist verify the precise spatial distribution of the pathology using the generated Grad-CAM heatmaps.")

        return {"diagnosis": condition, "confidence": round(final_conf, 4), "reasoning": reasoning}

if __name__ == "__main__":
    XRayFusionEngine().train_model()

⚙️ Training X-Ray Clinical Reasoner...
✅ X-Ray Fusion Model Saved.
